In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import jax
import jax.numpy as jnp
import numpy as np
from jax import random
import optax
from flax import serialization
from flax.training import train_state
import matplotlib.pyplot as plt
from dataset import *
from functools import partial
from tqdm import trange
from scipy.io import savemat, loadmat
import time
import orbax.checkpoint as ocp
from jax import tree_util
from jaxopt import LBFGS

data_config = DatasetConfig()
seed = 1234
key = jax.random.PRNGKey(seed)
np.random.seed(seed)

DEFAULT_DTYPE = data_config.dtype


@jax.jit
def mse(y_pre, y_true):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"
    return jnp.mean(jnp.square(y_pre - y_true))


@jax.jit
def weighted_mse(pred, target, alpha=10.0):
    assert pred.shape == target.shape, f"Shape mismatch: y_pre.shape = {pred.shape}, y_true.shape = {target.shape}"
    weights = jnp.where(jnp.abs(target) <= 5, alpha, 1.0)
    return jnp.mean(weights * (pred - target) ** 2)


@jax.jit
def improved_mse(y_pre, y_true):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"
    return jnp.mean((jnp.square(y_pre - y_true) + 1.0e-12)**(1/3))


def l2_relative_error(y_pre, y_true, dim=(1,2)):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"

    return jnp.linalg.norm(y_pre-y_true, ord=2, axis=dim) / jnp.linalg.norm(y_true, ord=2, axis=dim)


@partial(jax.jit, static_argnames='state_basis_forward')
def grad2_basis_f(x, params, state_basis_forward):
    def f(x_single):
        x_single = x_single[None, :]
        output = state_basis_forward(params, x_single)
        return output[0]

    def df_dxx(x):
        return jax.jacfwd(jax.jacfwd(f))(x)
    
    return jnp.squeeze(jax.vmap(lambda xi: df_dxx(xi))(x))


def create_train_state_deeponet(model, rng, example_input, lr, eta_min, betas=(0.9, 0.999), epochs=None):
    if isinstance(example_input, (tuple, dict)):
        params = model.init(rng, *example_input) if isinstance(example_input, tuple) else model.init(rng, **example_input)
    else:
        params = model.init(rng, example_input)

    if epochs is not None:
        scheduler = optax.cosine_decay_schedule(init_value=lr,
                                                decay_steps=epochs,
                                                alpha=eta_min/lr)
    else:
        scheduler = lr

    tx = optax.adamw(
        learning_rate=scheduler if epochs is not None else lr,
        b1=betas[0],
        b2=betas[1],
        eps=1e-8,
        weight_decay=1e-5
    )

    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx
    )


def create_train_state_transformer(model, rng, example_input, lr, eta_min,
                       betas=(0.9, 0.999),
                       epochs=None,
                       max_grad_norm=1.0,
                       warmup_steps=2000,
                      weight_decay=1e-6):
    if isinstance(example_input, (tuple, dict)):
        params = model.init(rng, *example_input) if isinstance(example_input, tuple) else model.init(rng, **example_input)
    else:
        params = model.init(rng, example_input)

    if epochs is not None:
        cosine = optax.cosine_decay_schedule(init_value=lr, decay_steps=epochs, alpha=eta_min / lr)
        schedule = optax.join_schedules(
            schedules=[optax.linear_schedule(init_value=1e-6, end_value=lr, transition_steps=warmup_steps),
                       cosine],
            boundaries=[warmup_steps]
        )
    else:
        schedule = lr

    transforms = []
    if max_grad_norm is not None:
        transforms.append(optax.clip_by_global_norm(max_grad_norm))
    transforms.append(optax.adamw(learning_rate=schedule, b1=betas[0], b2=betas[1], eps=1e-8, weight_decay=weight_decay))
    tx = optax.chain(*transforms)

    return train_state.TrainState.create(apply_fn=model.apply,
                                        params=params,
                                        tx=tx)


class MLP(nn.Module):
    dim: int
    ff_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.ff_dim)(x)
        x = jax.nn.gelu(x)
        x = nn.Dense(self.dim)(x)
        return x


# ========== Linear Self Attention ==========
class LinearAttention(nn.Module):
    dim: int
    num_heads: int
    attn_type: str = "l1"   # 可选: "l1", "galerkin", "l2"

    @nn.compact
    def __call__(self, x, y=None, add_identity: bool = True):
        """
        Linear Self-Attention or Cross-Attention
        x: [B, T1, dim]
        y: [B, T2, dim] (如果为 None，就是 Self-Attention)
        """
        y = x if y is None else y
        B, T1, C = x.shape
        _, T2, _ = y.shape

        assert C % self.num_heads == 0
        head_dim = C // self.num_heads

        # q, k, v projection
        q = nn.Dense(C, use_bias=False)(x).reshape(B, T1, self.num_heads, head_dim).transpose(0, 2, 1, 3)  # [B,H,T1,dh]
        k = nn.Dense(C, use_bias=False)(y).reshape(B, T2, self.num_heads, head_dim).transpose(0, 2, 1, 3)  # [B,H,T2,dh]
        v = nn.Dense(C)(y).reshape(B, T2, self.num_heads, head_dim).transpose(0, 2, 1, 3)  # [B,H,T2,dh]

        # 选择 φ(q), φ(k) 的变换方式
        if self.attn_type == "l1":
            q = jax.nn.softmax(q, axis=-1)
            k = jax.nn.softmax(k, axis=-1)
            k_cumsum = k.sum(axis=-2, keepdims=True)
            D_inv = 1.0 / (jnp.sum(q * k_cumsum, axis=-1, keepdims=True)) # 论文中的\alpha_t
        elif self.attn_type == "galerkin":
            q = jax.nn.softmax(q, axis=-1)
            k = jax.nn.softmax(k, axis=-1)
            D_inv = 1.0 / T2
        elif self.attn_type == "l2":
            q = q / (jnp.linalg.norm(q, ord=1, axis=-1, keepdims=True))
            k = k / (jnp.linalg.norm(k, ord=1, axis=-1, keepdims=True))
            k_cumsum = k.sum(axis=-2, keepdims=True)
            D_inv = 1.0 / (jnp.sum(jnp.abs(q * k_cumsum), axis=-1, keepdims=True))
        else:
            raise NotImplementedError(f"Unknown attn_type: {self.attn_type}")

        # context = kᵀ v
        context = jnp.einsum("bhnd,bhnv->bhdv", k, v)  # [B,H,dh,dh]

        # output
        y_out = jnp.einsum("bhnd,bhdv->bhnv", q, context)  # [B,H,T1,dh]
        y_out = y_out * D_inv

        if add_identity:
            y_out = y_out + q

        y_out = y_out.transpose(0, 2, 1, 3).reshape(B, T1, C)

        y_out = nn.Dense(C)(y_out)
        return y_out


# ========== Transformer Block ==========
class TransformerBlock(nn.Module):
    dim: int
    num_heads: int
    ff_dim: int

    @nn.compact
    def __call__(self, x):
        # LayerNorm + Linear Attention + Residual
        residual = x
        x = nn.LayerNorm()(x)
        attn_output = LinearAttention(self.dim, self.num_heads)(x)
        x = attn_output + residual

        # LayerNorm + MLP + Residual
        residual = x
        x = nn.LayerNorm()(x)
        x = MLP(self.dim, self.ff_dim)(x)
        x = x + residual

        return x


# ========== Transformer Encoder ==========
class TransformerEncoder(nn.Module):
    num_layers: int
    dim: int
    num_heads: int
    ff_dim: int

    @nn.compact
    def __call__(self, x):
        for _ in range(self.num_layers):
            x = TransformerBlock(self.dim, self.num_heads, self.ff_dim)(x)
        return x


class CoefficientPredictor(nn.Module):
    out_dim: int = data_config.basis_fn_dim
    f_token_num: int = 2
    bc_token_num: int = 2
    num_layers: int = 6
    dim: int = 512
    num_heads: int = 8
    ff_dim: int = 512*4

    @nn.compact
    def __call__(self, f_coef, u_bc):
        B = f_coef.shape[0]

        f_token = jnp.reshape(f_coef, (B, self.f_token_num, -1))  # [B, f_token_num, f_dim]
        f_token_emb = nn.Dense(self.dim)(f_token)  # [B, f_token_num, dim]

        bc_emb = jnp.reshape(u_bc, (B, self.bc_token_num, -1))
        bc_emb = nn.Dense(self.dim)(bc_emb)

        x = jnp.concatenate([f_token_emb, bc_emb], axis=1)  # [B, token_num, dim]
        
        token_num = self.f_token_num + self.bc_token_num
        pos_emb = self.param('pos_emb', nn.initializers.normal(0.02), (1, token_num, self.dim))
        x = x + pos_emb

        x = TransformerEncoder(self.num_layers, self.dim, self.num_heads, self.ff_dim)(x)

        x = nn.LayerNorm()(x)

        x = jnp.mean(x, axis=1)  # [B, dim]

        x = nn.Dense(self.out_dim)(x)
        return x


def save_state(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    checkpointer = ocp.PyTreeCheckpointer()
    checkpointer.save(path, state, force=True)


def restore_state(path, dummy_state):
    checkpointer = ocp.PyTreeCheckpointer()
    state = checkpointer.restore(path, item=dummy_state)
    return state


@jax.jit
def numerical_second_derivative_5point(f, x):
    h = x[1] - x[0]
    ddf = jnp.zeros_like(f)

    ddf = ddf.at[:, 2:-2].set(
        (-f[:, 4:] + 16 * f[:, 3:-1] - 30 * f[:, 2:-2] + 16 * f[:, 1:-3] - f[:, :-4]) / (12 * h ** 2)
    )

    ddf = ddf.at[:, 0].set((2 * f[:, 0] - 5 * f[:, 1] + 4 * f[:, 2] - f[:, 3]) / (h ** 2))
    ddf = ddf.at[:, 1].set((f[:, 0] - 2 * f[:, 1] + f[:, 2]) / (h ** 2))
    ddf = ddf.at[:, -2].set((f[:, -3] - 2 * f[:, -2] + f[:, -1]) / (h ** 2))
    ddf = ddf.at[:, -1].set((2 * f[:, -1] - 5 * f[:, -2] + 4 * f[:, -3] - f[:, -4]) / (h ** 2))

    return ddf


@partial(jax.jit, static_argnames='bnn')
def sample_params(sigmas_w, sigmas_b, keys, bnn):
    def single_sample(sigma_w, sigma_b, key):
        return bnn.hyper_initial(sigma_w, sigma_b, key)

    # vmap over sigma and key
    params_all = jax.vmap(single_sample)(sigmas_w, sigmas_b, keys)

    params_concat = tree_util.tree_map(lambda x: x.reshape((-1,) + x.shape[2:]), params_all)
    return params_concat

In [ ]:
DeepONet_config = DeepONetConfig()
xi_hidden_dim = DeepONet_config.branch.hidden_neuron
xi_output_dim = DeepONet_config.branch.output_neuron
xi_hidden_layers = DeepONet_config.branch.hidden_layers

mlp_xi = ResMLP(xi_hidden_dim, xi_output_dim, xi_hidden_layers, DeepONet_config.branch.activation_fn)
mlp_basis = TrunkNet(256, data_config.basis_fn_dim, 4, jax.nn.tanh, 14.0)

# 初始化模型参数
key, subkey = random.split(key)
xi_init = jnp.zeros((1, data_config.x_num))
trunknet_init = jnp.zeros((1, 1))

iterations = 150000
lr = 1.0e-4
eta_min_lr = 1.0e-6


key, key1, key2 = random.split(key, 3)
state_xi = create_train_state_deeponet(mlp_xi, key, xi_init, lr, eta_min_lr, epochs=iterations)
state_basis = create_train_state_deeponet(mlp_basis, key2, trunknet_init, lr, eta_min_lr, epochs=iterations)

state_xi = restore_state(os.path.join(data_config.checkpoint_dir, "deeponet_state_xi"), state_xi)
state_basis = restore_state(os.path.join(data_config.checkpoint_dir, "deeponet_state_basis"), state_basis)

In [ ]:
state_xi_forward = jax.jit(state_xi.apply_fn)
state_basis_forward = jax.jit(state_basis.apply_fn)

In [ ]:
x = data_config.x
basis = state_basis_forward(state_basis.params, x)
grad2_basis = grad2_basis_f(x, state_basis.params, state_basis_forward)

rank = np.linalg.matrix_rank(basis)
print(f"矩阵秩: {rank} / {data_config.basis_fn_dim}")

cond = np.linalg.cond(basis)
print(f"条件数: {cond}")


plt.figure()
plt.plot(x, basis)
plt.title('basis')
plt.show()

plt.figure()
plt.plot(x, grad2_basis)
plt.title('grad2_basis')
plt.show()

In [ ]:
# test deeponet
bnn_layer_size = [1, 100, 1]
bnn = BNN_sample(bnn_layer_size, 250)

sigma_list_w = data_config.sigma_list
sigma_list_b = sigma_list_w

keys = jax.random.split(key, len(sigma_list_w)+1)
key, subkey = keys[0], keys[1:]
params_concat = sample_params(sigma_list_w, sigma_list_b, subkey, bnn)

u_sin_test, d2u_dx2_sin_test = batch_compute_manual_derivatives_xt(x, params_concat, 0)
f_sin_test = f_func(u_sin_test, d2u_dx2_sin_test)

u_input = u_sin_test
f_input = f_sin_test

u_xi_test = state_xi_forward(state_xi.params, u_input)
u_pre = jnp.einsum('ik,jk->ij', u_xi_test, basis)

f_xi_test = state_xi_forward(state_xi.params, f_input)
f_pre = jnp.einsum('ik,jk->ij', f_xi_test, basis)
plt.figure()
plt.plot(u_input[:10, :].T, linestyle='-')
plt.plot(u_pre[:10, :].T, linestyle='--')
plt.show()

plt.figure()
plt.plot(f_input[:10, :].T, linestyle='-')
plt.plot(f_pre[:10, :].T, linestyle='--')
plt.title('f')
plt.show()


l2_error_u = l2_relative_error(u_pre, u_input, dim=(1))
l2_error_u_mean = jnp.mean(l2_error_u)
l2_error_u_std = jnp.std(l2_error_u)
print(f'l2_error_u mean={l2_error_u_mean:.3e}, l2_error_u std={l2_error_u_std:.3e}')

l2_error_f = l2_relative_error(f_pre, f_input, dim=(1))
l2_error_f_mean = jnp.mean(l2_error_f)
l2_error_f_std = jnp.std(l2_error_f)
print(f'l2_error_f mean={l2_error_f_mean:.3e}, l2_error_f std={l2_error_f_std:.3e}')

In [ ]:
transformer_model = CoefficientPredictor()

key, subkey = random.split(key)
f_xi_init = jnp.zeros((1, data_config.basis_fn_dim))
u_bc_init = jnp.zeros((1, 2, 2))

iterations = 200000
lr = 1.0e-4
eta_min_lr = 1.0e-7

key, subkey = random.split(key)
state_transformer = create_train_state_transformer(transformer_model, subkey, (f_xi_init, u_bc_init), lr, eta_min_lr, epochs=iterations)

In [ ]:
state_transformer = restore_state(os.path.join(data_config.checkpoint_dir, "transformer_linear"), state_transformer)

In [ ]:
state_transformer_forward = jax.jit(state_transformer.apply_fn)

In [ ]:
@jax.jit
def train_step(state_model, f_coef, u_bc, u_coef):
    def loss_fn(params):
        pred = state_model.apply_fn(params, f_coef, u_bc)
        return 10 * mse(pred, u_coef)

    loss, grads = jax.value_and_grad(loss_fn, argnums=(0))(state_model.params)
    state_model = state_model.apply_gradients(grads=grads)
    return state_model, loss

sigma_list_w = data_config.sigma_list
sigma_list_b = sigma_list_w
bnn_layer_size = [1, 100, 1]
batch_size = 100
bnn = BNN_sample(bnn_layer_size, batch_size)
x_in = jnp.tile(x[[0,-1], :][None, :, :], (batch_size*len(sigma_list_w), 1, 1))

loss_list = []

start_time = time.time()
for step in range(iterations):
    keys = random.split(key, len(sigma_list_w)+1)
    key, subkey = keys[0], keys[1:]
    params_all_concat = sample_params(sigma_list_w, sigma_list_b, subkey, bnn)

    u_sin_in, d2u_dx2_sin_in = batch_compute_manual_derivatives_xt(x, params_all_concat, 0)
    f_sin_in = f_func(u_sin_in, d2u_dx2_sin_in)

    u_input = u_sin_in
    f_input = f_sin_in

    u_xi_in = state_xi_forward(state_xi.params, u_input)
    f_xi_in = state_xi_forward(state_xi.params, f_input)
    u_bc = jnp.expand_dims(u_input[:, [0,-1]], axis=-1)
    u_bc_in = jnp.concatenate((u_bc, x_in), axis=-1)
    
    state_transformer, loss = train_step(state_transformer, f_xi_in, u_bc_in, u_xi_in)

    if (step+1) % 100 == 0:
        loss_list.append(loss)
        print(f"Step:[{step}/{iterations}], Loss: {loss:.3e}")

    if (step+1) % 1000 == 0:
        pred = state_transformer_forward(state_transformer.params, f_xi_in, u_bc_in)
        u_pre = jnp.einsum('ik,jk->ij', pred, basis)
        l2_error_u = jnp.mean(l2_relative_error(u_pre, u_input, dim=(1)))
        print(f"Testset error:{l2_error_u:.3e}")

end_time = time.time()
elapsed_time = end_time - start_time
print(f'Duration:{elapsed_time:.2f}s')

In [ ]:
print(elapsed_time)

In [ ]:
# test sin
test_num = 200

sigma_list_w = data_config.sigma_list
sigma_list_b = sigma_list_w
bnn_layer_size = [1, 100, 1]
bnn = BNN_sample(bnn_layer_size, test_num)

keys = random.split(key, len(sigma_list_w)+1)
key, subkey = keys[0], keys[1:]
params_all_concat = sample_params(sigma_list_w, sigma_list_b, subkey, bnn)

u_sin_test, d2u_dx2_sin_test = batch_compute_manual_derivatives_xt(x, params_all_concat, 0)
f_sin_test = f_func(u_sin_test, d2u_dx2_sin_test)

f_xi_test = state_xi_forward(state_xi.params, f_sin_test)
x_in = jnp.tile(x[[0,-1], :][None, :, :], (test_num*len(sigma_list_w), 1, 1))
u_bc = jnp.expand_dims(u_sin_test[:, [0,-1]], axis=-1)
u_bc_in = jnp.concatenate((u_bc, x_in), axis=-1)

pred = state_transformer_forward(state_transformer.params, f_xi_test, u_bc_in)

u_pre = jnp.einsum('ik,jk->ij', pred, basis)

l2_error_u = l2_relative_error(u_pre, u_sin_test, dim=(1))
l2_error_u_mean = jnp.mean(l2_error_u)
l2_error_u_std = jnp.std(l2_error_u)
print(f'l2_error_u_mean={l2_error_u_mean:.3e}, l2_error_u_std={l2_error_u_std:.3e}')


f_pre = jnp.einsum('ik,jk->ij', f_xi_test, basis)

l2_error_f = l2_relative_error(f_pre, f_sin_test, dim=(1))
l2_error_f_mean = jnp.mean(l2_error_f)
l2_error_f_std = jnp.std(l2_error_f)
print(f'l2_error_f mean={l2_error_f_mean:.3e}, l2_error_f std:{l2_error_f_std:.3e}')


for i, sigma in enumerate(sigma_list_w):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(u_sin_test[i*test_num:i*test_num+10, :].T, linestyle='-')
    plt.plot(u_pre[i*test_num:i*test_num+10, :].T, linestyle='--')
    plt.title(f"sigma={int(sigma)}, u")

    plt.subplot(1,2,2)
    plt.plot(f_sin_test[i*test_num:i*test_num+10, :].T, linestyle='-')
    plt.plot(f_pre[i*test_num:i*test_num+10, :].T, linestyle='--')
    plt.title(f"sigma={int(sigma)}, f")
    plt.tight_layout()
    plt.show()

    l2_error_u = l2_relative_error(u_pre[i*test_num:i*test_num+10, :], u_sin_test[i*test_num:i*test_num+10, :], dim=(1))
    l2_error_u_mean = jnp.mean(l2_error_u)
    l2_error_u_std = jnp.std(l2_error_u)
    print(f'l2_error_u_mean={l2_error_u_mean:.3e}, l2_error_u_std={l2_error_u_std:.3e}')

    l2_error_f = l2_relative_error(f_pre[i*test_num:i*test_num+10, :], f_sin_test[i*test_num:i*test_num+10, :], dim=(1))
    l2_error_f_mean = jnp.mean(l2_error_f)
    l2_error_f_std = jnp.std(l2_error_f)
    print(f'l2_error_f mean={l2_error_f_mean:.3e}, l2_error_f std:{l2_error_f_std:.3e}')

In [ ]:
save_state(state_transformer, os.path.join(data_config.checkpoint_dir, "transformer_linear"))

In [ ]:
def loss_fn(ft_input, basis, basis_dxx, u_bc, f):
    u_pre    = jnp.einsum('bk,xk->bx', ft_input, basis)
    u_pre_xx = jnp.einsum('bk,xk->bx', ft_input, basis_dxx)

    f_pre = f_func(u_pre, u_pre_xx)

    loss1 = mse(f_pre, f)
    loss2 = mse(u_pre[:, (0, -1)], u_bc)

    return 10 * loss1 + 100 * loss2


ft_input = pred                
u_bc = u_sin_test[:, (0, -1)]   
f_in = f_sin_test                     
x = x_in

solver = LBFGS(
    fun=loss_fn,
    maxiter=1,                 
    history_size=10,
    stepsize=1.0,              
    linesearch="zoom",         
    jit=True,                  
)

state = solver.init_state(
    ft_input,
    basis=basis, basis_dxx=grad2_basis,
    u_bc=u_bc, f=f_in
)


iterations_ft = 100
loss_list = []

for i in range(iterations_ft):
    ft_input, state = solver.update(
        ft_input, state,
        basis=basis, basis_dxx=grad2_basis,
        u_bc=u_bc, f=f_in
    )
    
    cur_loss = loss_fn(ft_input, basis, grad2_basis, u_bc, f_in)
    loss_list.append(float(cur_loss))
    if (i + 1) % 10 == 0:
        print(f"Step:[{i}/{iterations_ft}], loss:{cur_loss:.3e}")


plt.figure()
plt.plot(loss_list)
plt.yscale("log")
plt.title("Loss Curve (JAX LBFGS)")
plt.show()


u_pre_transformer = jnp.einsum('bk,xk->bx', ft_input, basis)
u_pre_np = np.array(u_pre_transformer)                           

index = np.random.randint(0, u_pre_np.shape[0], size=(10,))

plt.figure(figsize=(10, 4))
plt.plot(np.array(u_sin_test[index]).T, linestyle='-')
plt.plot(u_pre_np[index].T, linestyle='--')
plt.xlabel('t'); plt.ylabel('x')
plt.show()

U_true = jnp.array(u_sin_test)
U_pred = u_pre_transformer
print(U_true.shape)
print(U_pred.shape)
l2_error = jnp.linalg.norm(U_true - U_pred, ord=2, axis=(1)) / jnp.linalg.norm(U_true, ord=2, axis=(1))
l2_error_u_mean = jnp.mean(l2_error)
l2_error_u_std = jnp.std(l2_error)
print(f"l2_error_u={l2_error_u_mean:.3e}, l2_error_u_std={l2_error_u_std:.3e}")